In [1]:
import csv
import numpy as np
M = ["January","February","March","April","May","June","July","August",
     "September","October","November","December"]
C = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
rows = [[64,56,92,219,176,35,18,24,31,61,165,104],
        [35,42,70,145,120,78,96,105,72,55,60,38],
        [58,90,150,240,180,95,70,85,90,88,130,95],
        [30,15,55,150,240,110,90,65,60,90,100,70]]
raw = ["48","35","88","N/A","140"," 20 ","9","12","","45","182","96"]

def clean(r):
    v, bad = [], []
    for i, x in enumerate(r):
        try:
            v.append(int(x))
        except ValueError:
            v.append(None); bad.append(M[i])
    a = round(np.mean([x for x in v if x is not None]))
    return [a if x is None else x for x in v], bad, a

mach, bad, a = clean(raw)
print("Imputed:", bad, "=", a, "mm")

Imputed: ['April', 'September'] = 68 mm


In [2]:
R = np.array(rows + [mach])
print(R.shape, R.dtype)
tot = R.sum(axis=1)
print(dict(zip(C, tot.tolist())))
print(dict(zip(M, R.mean(axis=0).round(1).tolist())))
wet = np.array(M)[R.argmax(axis=1)]
for c, m in zip(C, wet): print(c, m)
dry = (R < 50).sum(axis=1)
S = R[:, :-1] + R[:, 1:]
k = S.argmax(axis=1)
win = np.char.add(np.char.add(np.array(M)[k], "-"), np.array(M)[k+1])
print(dry, win)

(5, 12) int64
{'Nairobi': 1045, 'Nakuru': 916, 'Kisumu': 1371, 'Mombasa': 1075, 'Machakos': 811}
{'January': 47.0, 'February': 47.6, 'March': 91.0, 'April': 164.4, 'May': 171.2, 'June': 67.6, 'July': 56.6, 'August': 58.2, 'September': 64.2, 'October': 67.8, 'November': 127.4, 'December': 80.6}
Nairobi April
Nakuru April
Kisumu April
Mombasa May
Machakos November
[4 3 0 2 6] ['April-May' 'April-May' 'April-May' 'April-May' 'November-December']


In [3]:
with open("rainfall_summary.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["county","annual_total_mm","wettest_month","dry_months","planting_window"])
    w.writerows(zip(C, tot.tolist(), wet.tolist(), dry.tolist(), win.tolist()))

## Planting Advisory
Plant in April-May in Nairobi, Nakuru, Kisumu and Mombasa, when two-month rainfall peaks (265-420 mm). Plant in November-December in Machakos, whose long rains are weak. Kisumu is wettest (1371 mm a year, no dry months). Machakos has six dry months, so use drought-tolerant seed and conserve water. Send SMS alerts a week before each window. Machakos figures include two estimated months, so confirm them with the station before relying on this advice.

## Data Quality Notes
Machakos was hand-typed. April ("N/A") and September (blank) were imputed at 68 mm, the mean of 10 valid values, so its totals and dry months are estimates. Verify with the station.